In [20]:
from google.cloud import bigquery

PROJECT = "gridzero-489711"
DATASET = "merged_set"
TABLE = "full_feature_engineered_data_test"

query = f"""
    SELECT *
    FROM {PROJECT}.{DATASET}.{TABLE}
"""

client = bigquery.Client('gridzero-489711')
query_job = client.query(query)
result = query_job.result()
df1 = result.to_dataframe()

/home/joseph/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
import joblib
from sklearn.preprocessing import MinMaxScaler

# 1. Define the exact column lists (Order is critical!)
feature_cols = [
    'temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms',
    'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2',
    'diffuse_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos',
    'biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar',
    'wind_offshore', 'wind_onshore'
]

target_cols = [
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

# 2. Fit the scalers on your BigQuery dataframe
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

# Important: ensure no NaN values before fitting, or it will error
# df_clean = df.dropna(subset=feature_cols + target_cols)

X_scaler.fit(df1[feature_cols])
y_scaler.fit(df1[target_cols])

# 3. Save to local .pkl files
joblib.dump(X_scaler, 'JM_lstm_x_scaler.pkl')
joblib.dump(y_scaler, 'JM_lstm_y_scaler.pkl')

print("✅ Scalers created and saved locally.")

✅ Scalers created and saved locally.


In [18]:
print("--- EXACT COLUMN ORDER ---")
print(list(X_scaler.feature_names_in_))

--- EXACT COLUMN ORDER ---
['temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar', 'wind_offshore', 'wind_onshore']


In [7]:
from google.cloud import storage

def upload_blob(bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"File {source_file_name} uploaded to {destination_blob_name}.")

# Replace with your bucket name from config.py
MY_BUCKET = "grid_zero_bucket"

upload_blob(MY_BUCKET, 'JM_lstm_x_scaler.pkl', 'weights/JM_lstm_x_scaler.pkl')
upload_blob(MY_BUCKET, 'JM_lstm_y_scaler.pkl', 'weights/JM_lstm_y_scaler.pkl')

File JM_lstm_x_scaler.pkl uploaded to weights/JM_lstm_x_scaler.pkl.
File JM_lstm_y_scaler.pkl uploaded to weights/JM_lstm_y_scaler.pkl.


In [15]:
import requests
import pandas as pd
import numpy as np


def fetch_forecast(latitude=51.5, longitude=-0.1):#target_date

    url = "https://api.open-meteo.com/v1/forecast"

    #start_str = target_date.strftime("%Y-%m-%d") if hasattr(target_date, 'strftime') else target_date

    hourly_vars = [
        "temperature_2m",
        "wind_gusts_10m",
        "cloud_cover",
        "direct_radiation",
        "diffuse_radiation",
        "shortwave_radiation",
        "wind_speed_120m",
        "wind_speed_80m",
        "pressure_msl",
        "precipitation",
    ]

    params = {
        "latitude": 51.5,
        "longitude": -0.1,
        "hourly": ",".join(hourly_vars),
        "timezone": "GMT",
        "forecast_days": 14,
        #"start_date": start_str,
        #"end_date": start_str,
        "wind_speed_unit": "ms",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if "hourly" not in data:
        raise ValueError(f"Unexpected API response: {data}")

    df = pd.DataFrame(data["hourly"])

    return df

In [16]:
df = fetch_forecast()

In [21]:
df1.shape

(148991, 31)

In [23]:
df1

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,...,carbon_intensity_gco2_kwh,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520
0,2017-09-12 00:00:00+00:00,11.6,31.0,28.1,4.0,0.0,0.0,0.0,1001.2,0.0,...,142.000000,0.000000,1.000000,0.781831,0.623490,-0.948362,-0.317191,NaN,NaN,NaN
1,2017-09-12 00:30:00+00:00,11.6,31.0,28.1,4.0,0.0,0.0,0.0,1001.2,0.0,...,140.000000,0.000000,1.000000,0.781831,0.623490,-0.948362,-0.317191,NaN,NaN,NaN
2,2017-09-12 01:00:00+00:00,11.2,30.3,27.0,5.0,0.0,0.0,0.0,1001.9,0.0,...,139.000000,0.258819,0.965926,0.781831,0.623490,-0.948362,-0.317191,NaN,NaN,NaN
3,2017-09-12 01:30:00+00:00,11.2,30.3,27.0,5.0,0.0,0.0,0.0,1001.9,0.0,...,137.000000,0.258819,0.965926,0.781831,0.623490,-0.948362,-0.317191,NaN,NaN,NaN
4,2017-09-12 02:00:00+00:00,10.9,29.6,25.2,7.0,0.0,0.0,0.0,1002.4,0.0,...,132.000000,0.500000,0.866025,0.781831,0.623490,-0.948362,-0.317191,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148986,2026-03-12 21:00:00+00:00,11.4,50.8,68.0,100.0,0.0,0.0,0.0,1004.1,0.0,...,77.371428,-0.707107,0.707107,0.433884,-0.900969,0.939856,0.341571,76.0,152.0,221.0
148987,2026-03-12 21:30:00+00:00,11.4,50.8,68.0,100.0,0.0,0.0,0.0,1004.1,0.0,...,77.371428,-0.707107,0.707107,0.433884,-0.900969,0.939856,0.341571,87.0,147.0,200.0
148988,2026-03-12 22:00:00+00:00,11.4,51.5,67.3,100.0,0.0,0.0,0.0,1003.0,0.1,...,77.371428,-0.500000,0.866025,0.433884,-0.900969,0.939856,0.341571,77.0,147.0,182.0
148989,2026-03-12 22:30:00+00:00,11.4,51.5,67.3,100.0,0.0,0.0,0.0,1003.0,0.1,...,77.371428,-0.500000,0.866025,0.433884,-0.900969,0.939856,0.341571,78.0,149.0,168.0


In [24]:
from google.cloud import bigquery
import pandas as pd

project_id = "gridzero-489711"
dataset_id = "merged_set"
table_id = "full_feature_engineered_data_test"

def fetch_historical_gen(project_id, dataset_id, table_id):
    client = bigquery.Client(project=project_id)

    # Query the last 7 days of generation data
    query = f"""
        SELECT
            timestamp as time,
            biomass, fossil_gas, fossil_hard_coal, hydro_pumped_storage,
            hydro_run_of_river_and_poundage, nuclear, other, solar,
            wind_offshore, wind_onshore
        FROM `{project_id}.{dataset_id}.{table_id}`
        WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
        ORDER BY time ASC
    """

    df_gen = client.query(query).to_dataframe()

    # Resample to 30min and forward-fill to match weather prep
    df_gen['time'] = pd.to_datetime(df_gen['time'])
    df_gen = df_gen.set_index('time').resample('30min').ffill().reset_index()

    return df_gen

In [25]:
fetch_historical_gen(project_id = "gridzero-489711",
dataset_id = "merged_set",
table_id = "full_feature_engineered_data_test")

BadRequest: 400 Unrecognized name: timestamp at [8:15]; reason: invalidQuery, location: query, message: Unrecognized name: timestamp at [8:15]

Location: EU
Job ID: c6bcb8a1-ade2-4dae-b280-2c7cbcf2da3d
